# DBSP operators

Even more than the weighted operators, you should usually not need to care about the *DBSP operators*. With them, you also venture through the abstraction that Kafi Streams has built around DBSP/pydbsp for developer experience.

The "low-level" aspect of these operators is reflected in their `_` prefix.

In the examples, we'll change the way how we push inputs and receive outputs: We explicitly associate a weight (`1`) with each input record, and let Kafi Streams also associate its returned records with their respective weights.


## Overview

[Preparation](#prep)

* [_integrate()](#_integrate-operator)
* [_differentiate()](#_differentiate-operator)
* [_delay()](#_delay-operator)


---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [1]:
!pip install -r ../requirements.txt

import sys
sys.path.insert(1, "../")
sys.path.insert(1, "../../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import ClickGenerator, CustomerGenerator
click_generator = ClickGenerator()
customer_generator = CustomerGenerator()

click_source_str = "clicks"
customer_source_str = "customers"



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Please also note that when we re-use the same example over and over again to illustrate how the operators work, we always mark the important new parts as follows:
```python
    # <------------------------------>
    ...important new parts...
    # <------------------------------>
```

<a id="_integrate-operator"></a>
## _integrate()

The `_integrate()` operator is the direct equivalent of DBSPs `integrate()` operator. It computes the running sum (accumulation) of changes over time.

```
_integrate(**kwargs)
```

Here is an example.

In [ ]:
built_tn = Tn.build(
    Tn.source(click_source_str).to_zSet(Tn._from_records)
    #
    # <------------------------------>
    ._integrate()
    # <------------------------------>
).from_zSet(Tn._to_records)

print("Step 1")

input_m_w_tuple_list = [(m, 1) for m in click_generator.generate(3)]
print("Input:")
for m_w_tuple in input_m_w_tuple_list:
    print(m_w_tuple)

print("Output")
output_m_w_tuple_list = built_tn.process({click_source_str: input_m_w_tuple_list})
for m_w_tuple in output_m_w_tuple_list:
    print(m_w_tuple)

#

print("\nStep 2")

print("Input:")
for m_w_tuple in input_m_w_tuple_list:
    print(m_w_tuple)

print("Output:")
output_m_w_tuple_list = built_tn.process({click_source_str: input_m_w_tuple_list})
for m_w_tuple in output_m_w_tuple_list:
    print(m_w_tuple)


Step 1
Input:
({'key': None, 'value': {'customer_id': 52, 'view_time': 86, 'ts': 1786709537121}}, 1)
({'key': None, 'value': {'customer_id': 90, 'view_time': 72, 'ts': 1786709537221}}, 1)
({'key': None, 'value': {'customer_id': 22, 'view_time': 41, 'ts': 1786709537321}}, 1)
Output
({'key': None, 'value': {'customer_id': 52, 'view_time': 86, 'ts': 1786709537121}}, 1)
({'key': None, 'value': {'customer_id': 90, 'view_time': 72, 'ts': 1786709537221}}, 1)
({'key': None, 'value': {'customer_id': 22, 'view_time': 41, 'ts': 1786709537321}}, 1)

Step 2
Input:
({'key': None, 'value': {'customer_id': 52, 'view_time': 86, 'ts': 1786709537121}}, 1)
({'key': None, 'value': {'customer_id': 90, 'view_time': 72, 'ts': 1786709537221}}, 1)
({'key': None, 'value': {'customer_id': 22, 'view_time': 41, 'ts': 1786709537321}}, 1)
Output
({'key': None, 'value': {'customer_id': 52, 'view_time': 86, 'ts': 1786709537121}}, 2)
({'key': None, 'value': {'customer_id': 90, 'view_time': 72, 'ts': 1786709537221}}, 2)


In step 1, `_integrate()` outputs the input records with their input weight `1`.

In step 2, when we push the same records again, `_integrate()` accumulates their weights to `2`.


<a id="_differentiate-operator"></a>
## _differentiate()

The `_differentiate()` operator is the direct equivalent of DBSPs `integrate()` operator. It is the inverse of `_integrate()`: it computes the difference (delta) between the current state and the previous step's state. It returns only what has changed since the last step.

```
_differentiate(**kwargs)
```

Here is an example.

In [ ]:
built_tn = Tn.build(
    Tn.source(click_source_str).to_zSet(Tn._from_records)
    #
    # <------------------------------>
    ._differentiate()
    # <------------------------------>
).from_zSet(Tn._to_records)

print("Step 1")

input_m_w_tuple_list = [(m, 1) for m in click_generator.generate(3)]
print("Input:")
for m_w_tuple in input_m_w_tuple_list:
    print(m_w_tuple)

print("Output")
output_m_w_tuple_list = built_tn.process({click_source_str: input_m_w_tuple_list})
for m_w_tuple in output_m_w_tuple_list:
    print(m_w_tuple)

#

print("\nStep 2")

print("Input:")
for m_w_tuple in input_m_w_tuple_list:
    print(m_w_tuple)

print("Output:")
output_m_w_tuple_list = built_tn.process({click_source_str: input_m_w_tuple_list})
for m_w_tuple in output_m_w_tuple_list:
    print(m_w_tuple)


Step 1
Input:
({'key': None, 'value': {'customer_id': 93, 'view_time': 58, 'ts': 1786709537421}}, 1)
({'key': None, 'value': {'customer_id': 33, 'view_time': 86, 'ts': 1786709537521}}, 1)
({'key': None, 'value': {'customer_id': 18, 'view_time': 29, 'ts': 1786709537621}}, 1)
Output
({'key': None, 'value': {'customer_id': 93, 'view_time': 58, 'ts': 1786709537421}}, 1)
({'key': None, 'value': {'customer_id': 33, 'view_time': 86, 'ts': 1786709537521}}, 1)
({'key': None, 'value': {'customer_id': 18, 'view_time': 29, 'ts': 1786709537621}}, 1)

Step 2
Input:
({'key': None, 'value': {'customer_id': 93, 'view_time': 58, 'ts': 1786709537421}}, 1)
({'key': None, 'value': {'customer_id': 33, 'view_time': 86, 'ts': 1786709537521}}, 1)
({'key': None, 'value': {'customer_id': 18, 'view_time': 29, 'ts': 1786709537621}}, 1)
Output


DBSP uses `_differentiate()` to determine the changes (deltas) of its internal state.

In the example, as we do not push any *new* data in step 2 but the same as in step 1, the second step does not result in any outputs:  the integrated state did not change.


<a id="_delay-operator"></a>
## _delay()

The `_delay()` operator is the direct equivalent of DBSPs `delay()` operator. It basically stores the inputs from the last step in memory.
```
_delay(**kwargs)
```

Here is an example.

In [8]:
built_tn = Tn.build(
    Tn.source(click_source_str).to_zSet(Tn._from_records)
    #
    # <------------------------------>
    ._delay()
    ._integrate()
    # <------------------------------>
).from_zSet(Tn._to_records)

print("Step 1")

input_m_w_tuple_list = [(m, 1) for m in click_generator.generate(3)]
print("Input:")
for m_w_tuple in input_m_w_tuple_list:
    print(m_w_tuple)

print("Output")
output_m_w_tuple_list = built_tn.process({click_source_str: input_m_w_tuple_list})
for m_w_tuple in output_m_w_tuple_list:
    print(m_w_tuple)

#

print("\nStep 2")

input_m_w_tuple_list = [(m, 1) for m in click_generator.generate(3)]
print("Input:")
for m_w_tuple in input_m_w_tuple_list:
    print(m_w_tuple)

print("Output:")
output_m_w_tuple_list = built_tn.process({click_source_str: input_m_w_tuple_list})
for m_w_tuple in output_m_w_tuple_list:
    print(m_w_tuple)


Step 1
Input:
({'key': None, 'value': {'customer_id': 62, 'view_time': 45, 'ts': 1786712827584}}, 1)
({'key': None, 'value': {'customer_id': 27, 'view_time': 30, 'ts': 1786712827684}}, 1)
({'key': None, 'value': {'customer_id': 96, 'view_time': 60, 'ts': 1786712827784}}, 1)
Output
({'key': None, 'value': {'customer_id': 62, 'view_time': 45, 'ts': 1786712827584}}, 1)
({'key': None, 'value': {'customer_id': 27, 'view_time': 30, 'ts': 1786712827684}}, 1)
({'key': None, 'value': {'customer_id': 96, 'view_time': 60, 'ts': 1786712827784}}, 1)

Step 2
Input:
({'key': None, 'value': {'customer_id': 36, 'view_time': 30, 'ts': 1786712827884}}, 1)
({'key': None, 'value': {'customer_id': 98, 'view_time': 74, 'ts': 1786712827984}}, 1)
({'key': None, 'value': {'customer_id': 3, 'view_time': 16, 'ts': 1786712828084}}, 1)
Output:
({'key': None, 'value': {'customer_id': 36, 'view_time': 30, 'ts': 1786712827884}}, 1)
({'key': None, 'value': {'customer_id': 98, 'view_time': 74, 'ts': 1786712827984}}, 1)


We make the effect of `_delay()` visible by adding `_integrate()` behind it to "integrate" all the accumulated inputs so far.

So what happens in the example:
1. After the first step, `_delay()` has only accumulated the first input.
2. After the second step, `_delay()` has accumulated the first and the second input.
